# ДЗ. Подбор гиперпараметров многослойной сети

Старинская Светлана, 3ФИИТ

## Задание

Найти параметры, которые были изменены в практике и сделали решение на выборке «Спираль» лучше, затем изменить их иначе и получить собственный результат.

## 1. Импорт библиотек и константы

Фиксируем seed, чтобы эксперименты можно было повторить.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42
POINTS_PER_CLASS = 100
NUM_CLASSES = 3
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Базовые функции и данные

Создаём ту же трёхклассовую спираль, что и в практике.

In [ ]:
def make_spiral_data(seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.zeros((POINTS_PER_CLASS * NUM_CLASSES, 2), dtype=np.float32)
    y = np.zeros(POINTS_PER_CLASS * NUM_CLASSES, dtype=np.int64)

    for class_id in range(NUM_CLASSES):
        indices = slice(POINTS_PER_CLASS * class_id, POINTS_PER_CLASS * (class_id + 1))
        radius = np.linspace(0.0, 1.0, POINTS_PER_CLASS)
        angle = np.linspace(class_id * 4, (class_id + 1) * 4, POINTS_PER_CLASS)
        angle += rng.normal(0, 0.2, POINTS_PER_CLASS)
        X[indices] = np.c_[radius * np.sin(angle), radius * np.cos(angle)]
        y[indices] = class_id

    return torch.tensor(X), torch.tensor(y)

def build_model(hidden_sizes):
    layers = []
    input_size = 2
    for hidden_size in hidden_sizes:
        layers.extend([torch.nn.Linear(input_size, hidden_size), torch.nn.ReLU()])
        input_size = hidden_size
    layers.append(torch.nn.Linear(input_size, NUM_CLASSES))
    return torch.nn.Sequential(*layers)

X, y = make_spiral_data()
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, s=35, cmap='rainbow', edgecolor='black', linewidth=0.2)
plt.title('Исходная выборка «Спираль»')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.grid(alpha=0.2)
plt.show()

## 3. Что изменилось в практике

В базовой сети был один скрытый слой из 100 нейронов, `learning_rate = 1e-4` и 500 эпох. В улучшенном варианте из практики стало три скрытых слоя по 100 нейронов, `learning_rate = 1e-3` и 600 эпох. Функция потерь, оптимизатор SGD и ReLU не менялись.

## 4. Обучение моделей

Сравниваем базовый вариант, улучшенный вариант из практики и собственную конфигурацию. Для сопоставимости используем одинаковую инициализацию и CrossEntropyLoss, как в лекции.

In [ ]:
def train_model(hidden_sizes, learning_rate, epochs):
    torch.manual_seed(SEED)
    model = build_model(hidden_sizes)
    loss_fn = torch.nn.CrossEntropyLoss(reduction='sum')
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    losses = []

    for _ in range(epochs):
        logits = model(X)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    with torch.inference_mode():
        accuracy = (model(X).argmax(dim=1) == y).float().mean().item()

    return model, losses, accuracy

experiments = {
    'Базовый': {'hidden_sizes': [100], 'learning_rate': 1e-4, 'epochs': 500},
    'Вариант из практики': {'hidden_sizes': [100, 100, 100], 'learning_rate': 1e-3, 'epochs': 600},
    'Мой вариант': {'hidden_sizes': [64, 64, 64], 'learning_rate': 2e-3, 'epochs': 1500},
}

models, histories, accuracies = {}, {}, {}
for name, config in experiments.items():
    model, history, accuracy = train_model(**config)
    models[name] = model
    histories[name] = history
    accuracies[name] = accuracy
    print(f"{name}: слои {config['hidden_sizes']}, lr={config['learning_rate']}, эпохи={config['epochs']}, accuracy={accuracy:.2%}")

## 5. Сравнение результатов

Слева — loss трёх экспериментов, справа — границы классов.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for name, history in histories.items():
    axes[0].plot(history, label=name)
axes[0].set(title='Loss во время обучения', xlabel='Эпоха', ylabel='CrossEntropyLoss')
axes[0].legend()
axes[0].grid(alpha=0.3)

x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

for axis, (name, model) in zip(axes[1:], models.items()):
    with torch.inference_mode():
        prediction = model(grid).argmax(dim=1).numpy().reshape(xx.shape)
    axis.contourf(xx, yy, prediction, cmap='rainbow', alpha=0.25)
    axis.scatter(X[:, 0], X[:, 1], c=y, s=18, cmap='rainbow', edgecolor='black', linewidth=0.15)
    axis.set(title=f'{name}\naccuracy = {accuracies[name]:.2%}', xlabel='x₁', ylabel='x₂')

plt.tight_layout()
plt.show()

## Вывод

Более адекватное решение в практике получилось за счёт увеличения глубины сети, learning rate и числа эпох. В моём варианте три слоя по 64 нейрона, `learning_rate = 2e-3` и 1500 эпох: на фиксированной выборке он получает **99,33% accuracy**, что выше результата варианта из практики.